# **Processing MedQA dataset**
The original dataset did not include rationales; we are using the dataset from PMC-LLaMA.

## **Verification of the existence of repeated questions in the initial dataset**

This script analyzes the raw dataset to check for duplicated questions (based on `sample_id`) and evaluates the completeness of the generated responses. 

- Calculate how many unique questions exist and how many times each question is repeated.
- Count the total number of entries that include a valid medical explanation (`###Rationale`) versus those that lack one.

**Observation:**
The analysis shows that variations exist. There are entries without a rationale, and typically one to two versions of the same question with different rationales.

In [2]:
import json
import os
from collections import Counter

# --- CONFIGURATION ---
INPUT_FILENAME = "../../../Data/MedC_I_PMCllama/MedQA/medqa_data.json"

def analyze_id_frequency():
    print(f"--- Frequency of sample_id in {INPUT_FILENAME} ---")
    
    if not os.path.exists(INPUT_FILENAME):
        print(f"ERROR: File '{INPUT_FILENAME}' doesn't exist.")
        return

    found_ids = []
    total_lines = 0
    with_rationale_count = 0
    without_rationale_count = 0

    try:
        with open(INPUT_FILENAME, 'r', encoding='utf-8') as infile:
            for line in infile:
                line = line.strip()
                if not line: continue
                
                total_lines += 1
                try:
                    entry = json.loads(line)
                    
                    # Save the ID in the list
                    found_ids.append(entry.get("sample_id"))
                    
                    # Check if the entry has a rationale
                    output_text = entry.get("output", "")
                    if "###Rationale" in output_text:
                        with_rationale_count += 1
                    else:
                        without_rationale_count += 1
                        
                except json.JSONDecodeError:
                    continue

        # Count how many times each ID appears
        count_by_id = Counter(found_ids)

        # Count the distribution of these frequencies
        # E.g.: How many IDs appear 3 times or 2 times?
        distribution = Counter(count_by_id.values())

        print("\n" + "-"*40)
        print(" SUMMARY")
        print("-" * 40)
        print(f"Total lines processed:       {total_lines}")
        print(f"Total unique IDs:            {len(count_by_id)}")
        print(f"Entries WITH rationale:      {with_rationale_count}")
        print(f"Entries WITHOUT rationale:   {without_rationale_count}")
        print("-" * 40)
        print(" DISTRIBUTION:")
        
        # Sort by number of occurrences (1, 2, 3...)
        for num_times in sorted(distribution.keys()):
            qty_ids = distribution[num_times]
            print(f"IDs that show {num_times} time(s): {qty_ids}")
        
        print("-" * 40)

        # Verify if all questions have exactly 3 versions of response
        if distribution.get(3) == len(count_by_id):
            print("Every question has exactly 3 versions of response.")
        else:
            print("Variations exist. Not every question has 3 versions of response.")

    except Exception as e:
        print(f"Error: {e}")

analyze_id_frequency()

--- Frequency of sample_id in ../../../Data/MedC_I_PMCllama/MedQA/medqa_data.json ---

----------------------------------------
 SUMMARY
----------------------------------------
Total lines processed:       20791
Total unique IDs:            10177
Entries WITH rationale:      10614
Entries WITHOUT rationale:   10177
----------------------------------------
 DISTRIBUTION:
IDs that show 2 time(s): 9740
IDs that show 3 time(s): 437
----------------------------------------
Variations exist. Not every question has 3 versions of response.


## **MedQA Training Set Generation**

This script processes the raw MedQA dataset to create a clean, instruction-tuned dataset (`medqa_train.jsonl`) suitable for fine-tuning.

- **Filtering:** Selects only entries that contain a generated "###Rationale" in the output.
- **Instruction Injection:** Augments each example with a randomly selected system instruction from a predefined pool.
- **Formatting:** Standardizes the data strictly into `instruction`, `input`, and `output` fields, stripping away unnecessary metadata, and streams it directly into a JSONL format.

In [ ]:
import json
import os
import random

# --- CONFIGURATION ---
INPUT_FILENAME = "../../../Data/MedC_I_PMCllama/MedQA/medqa_data.json"
OUTPUT_FILENAME = "medqa_train.jsonl"

INSTRUCTIONS_POOL = [
    "Analyze this clinical scenario as a decision support assistant. Justify your reasoning through medical evidence and select the correct answer.",
    "Review the following clinical scenario. Provide a step-by-step medical explanation for your conclusion and pick the best answer.",
    "As an expert medical assistant, provide a thorough analysis of the following case. Offer a clinical rationale before stating the final answer.",
    "Review the provided medical information as a decision support system. Develop a clinical justification and identify the correct response.",
    "Examine the following clinical scenario to assist a medical professional. Provide a rationale and state the final outcome."
]

def process_medqa_to_jsonl():
    print(f"--- Processing MedQA to JSONL ---")
    
    if not os.path.exists(INPUT_FILENAME):
        print(f"ERROR: File '{INPUT_FILENAME}' not found.")
        return

    total_lines_read = 0
    total_entries_saved = 0

    try:
        # Open both files simultaneously to read and write on the fly
        with open(INPUT_FILENAME, 'r', encoding='utf-8') as infile, \
             open(OUTPUT_FILENAME, 'w', encoding='utf-8') as outfile:
            
            for line in infile:
                line = line.strip()
                if not line: continue
                
                total_lines_read += 1
                entry = json.loads(line)
                
                # 1. Keep ALL entries that contain a rationale
                if "###Rationale" in entry.get("output", ""):
                    
                    # 2. Create the final clean object 
                    # Exclude source and sample_id, and assign a random instruction
                    clean_entry = {
                        "instruction": random.choice(INSTRUCTIONS_POOL),
                        "input": entry.get("input", ""),
                        "output": entry.get("output", "")
                    }
                    
                    # Convert to JSON string in a single line and write
                    json_line = json.dumps(clean_entry, ensure_ascii=False)
                    outfile.write(json_line + "\n")
                    
                    total_entries_saved += 1

        # --- PRINT SUMMARY ---
        print(f"Total lines read: {total_lines_read}")
        print("-" * 40)
        print(f"Output file: {OUTPUT_FILENAME}")
        print(f"Total entries saved (all rationales kept): {total_entries_saved}")
        print("-" * 40)

    except Exception as e:
        print(f"Error: {e}")

process_medqa_to_jsonl()

--- Processing MedQA to JSONL ---
Total lines read: 20791
----------------------------------------
Output file: medqa_train.jsonl
Total entries saved (all rationales kept): 10614
----------------------------------------


## **Split the train dataset in train and dev datasets**

Since the original MedQA dev dataset doesn't have rationales, we divided the train dataset in train and dev

In [6]:
import json
import random
import os

def generate_medqa_splits(input_path, train_path, dev_path, dev_size=1000, seed=42):
    
    with open(input_path, 'r', encoding='utf-8') as f:
        lines = [line for line in f if line.strip()]
    
    random.seed(seed)
    random.shuffle(lines)

    dev_data = lines[:dev_size]
    train_data = lines[dev_size:]

    with open(dev_path, 'w', encoding='utf-8') as f:
        for line in dev_data:
            f.write(line)
    
    with open(train_path, 'w', encoding='utf-8') as f:
        for line in train_data:
            f.write(line)
            
    print(f"Success! MedQA splited:")
    print(f" -> TRAIN: {len(train_data)} entries in '{os.path.basename(train_path)}'")
    print(f" -> DEV: {len(dev_data)} entries in '{os.path.basename(dev_path)}'")

input_file = "MedQA_train.jsonl"
train_clean = "MedQA_train_final.jsonl"
dev_file = "MedQA_dev.jsonl"

generate_medqa_splits(input_file, train_clean, dev_file)

Success! MedQA splited:
 -> TRAIN: 9614 entries in 'MedQA_train_final.jsonl'
 -> DEV: 1000 entries in 'MedQA_dev.jsonl'


## **Change the instructions**

In [7]:
import json
import random
import os

INSTRUCTION_POOL = [
    "You are an expert Clinical Decision Support System assisting an attending physician. Analyze the clinical case using evidence-based medical reasoning, explain your logic, and identify the correct answer from the options provided.",
    "Act as an expert medical consultant supporting a physician's clinical analysis. Review the case presentation, justify your conclusion with current medical guidelines, and select the best answer. Maintain a professional and direct clinical tone.",
    "Provide clinical decision support to the treating physician. Analyze this clinical case using step-by-step medical reasoning and state the exact correct answer choice."
]

def apply_system_prompts_to_medqa(input_file, output_file):
    print(f"Processing: {input_file}...")
    
    if not os.path.exists(input_file):
        print(f"[ERRO] File not found: {input_file}")
        return

    processed_count = 0

    with open(input_file, 'r', encoding='utf-8') as infile, \
         open(output_file, 'w', encoding='utf-8') as outfile:
        
        for line in infile:
            if not line.strip():
                continue
                
            data = json.loads(line)
            
            system_context = random.choice(INSTRUCTION_POOL)
            
            data["instruction"] = system_context
            
            json.dump(data, outfile, ensure_ascii=False)
            outfile.write('\n')
            
            processed_count += 1

    print(f"[SUCCESS] {processed_count} entries saved in: {output_file}")

def main():
    random.seed(42) 
    
    train_input = "MedQA_train_final.jsonl"
    train_output = "MedQA_train_final_UPDATED.jsonl"
    
    dev_input = "MedQA_dev.jsonl"
    dev_output = "MedQA_dev_UPDATED.jsonl"
    
    apply_system_prompts_to_medqa(train_input, train_output)
    apply_system_prompts_to_medqa(dev_input, dev_output)
    

if __name__ == "__main__":
    main()

Processing: MedQA_train_final.jsonl...
[SUCCESS] 9614 entries saved in: MedQA_train_final_UPDATED.jsonl
Processing: MedQA_dev.jsonl...
[SUCCESS] 1000 entries saved in: MedQA_dev_UPDATED.jsonl
